# 03 · Main effects

**Goal:** estimate how much each email changed visits, conversions and spend over the two weeks after sending, with confidence intervals, and decide which effects are real after accounting for running several tests.

**The test family (9 tests):** three contrasts × three metrics.

| Contrast | Question |
|---|---|
| Mens − No E-Mail | Did the Mens email work? |
| Womens − No E-Mail | Did the Womens email work? |
| Mens − Womens | Which email worked better? |

Notebook 02 showed the arms are balanced and correctly sized, so a simple difference in means between two arms is an unbiased estimate of the effect. No regression adjustment is needed for unbiasedness; notebook 04 uses it only to reduce variance.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hillstrom.data import load_customers
from hillstrom.effects import bootstrap_diff_ci, diff_means_welch, diff_proportions, holm_adjust
from hillstrom.plots import ARM_COLORS, ARM_LABELS, TEXT_PRIMARY, TEXT_SECONDARY, apply_style, save_figure

apply_style()
df = load_customers()

CONTRASTS = [("mens", "control"), ("womens", "control"), ("mens", "womens")]


def outcome(arm: str, metric: str) -> np.ndarray:
    return df.loc[df["arm"] == arm, metric].to_numpy()


def contrast_name(treat: str, ctrl: str) -> str:
    return f"{ARM_LABELS[treat]} − {ARM_LABELS[ctrl]}"

## 1. Visit and conversion rates

Both are 0/1 outcomes, so the effect is a difference in proportions, reported in **percentage points (pp)**.

- **Confidence interval:** Wald, $\hat p_t - \hat p_c \pm 1.96\,\text{SE}$, with $\text{SE} = \sqrt{\hat p_t(1-\hat p_t)/n_t + \hat p_c(1-\hat p_c)/n_c}$.
- **p-value:** z-test with the same SE, so the interval excludes 0 exactly when p < 0.05.

Why Wald and not an exact or Newcombe interval: those matter for small samples or very few events. Here each arm has ~21,000 customers and even the rarest outcome (conversion, ~1%) has 120-270 events per arm, which is enough for the normal approximation.

In [2]:
rate_rows = []
for treat, ctrl in CONTRASTS:
    for metric in ["visit", "conversion"]:
        r = diff_proportions(outcome(treat, metric), outcome(ctrl, metric))
        rate_rows.append(
            {
                "contrast": contrast_name(treat, ctrl),
                "metric": metric,
                "rate_treat": r.mean_t,
                "rate_ctrl": r.mean_c,
                "diff_pp": 100 * r.estimate,
                "ci_low_pp": 100 * r.ci_low,
                "ci_high_pp": 100 * r.ci_high,
                "p_value": r.p_value,
            }
        )
rates = pd.DataFrame(rate_rows)
rates.style.format(
    {"rate_treat": "{:.2%}", "rate_ctrl": "{:.2%}", "diff_pp": "{:+.2f}",
     "ci_low_pp": "{:+.2f}", "ci_high_pp": "{:+.2f}", "p_value": "{:.2g}"}
).hide(axis="index")

contrast,metric,rate_treat,rate_ctrl,diff_pp,ci_low_pp,ci_high_pp,p_value
Mens E-Mail − No E-Mail,visit,18.28%,10.62%,+7.66,+7.00,+8.32,2.7e-113
Mens E-Mail − No E-Mail,conversion,1.25%,0.57%,+0.68,+0.50,+0.86,1.5e-13
Womens E-Mail − No E-Mail,visit,15.14%,10.62%,+4.52,+3.89,+5.16,1.9e-44
Womens E-Mail − No E-Mail,conversion,0.88%,0.57%,+0.31,+0.15,+0.47,0.00016
Mens E-Mail − Womens E-Mail,visit,18.28%,15.14%,+3.14,+2.43,+3.84,3.6e-18
Mens E-Mail − Womens E-Mail,conversion,1.25%,0.88%,+0.37,+0.17,+0.56,0.00021


**Reading the table (before the multiple-comparison correction in section 3):**

- **Visits:** the Mens email raised the two-week visit rate by about 7.7 pp (from 10.6% to 18.3%), the Womens email by about 4.5 pp. Mens beats Womens by about 3.1 pp. All intervals are narrow and far from 0.
- **Conversions:** the effects are small in absolute terms (+0.68 pp for Mens, +0.31 pp for Womens) because the base rate is only 0.57%. The intervals are wide relative to the estimates: for Womens the true effect could plausibly be anywhere from about +0.15 to +0.47 pp.